# Check MWDC pulser data
The input parquet file should be the output of `streamingv1_to_parquet.py`

In [4]:
from pyspark.sql import SparkSession,DataFrame,Window
from pyspark.sql import functions as F
import argparse
from pathlib import Path
import re

CH2NS = 0.0009765625 # AMANEQ HRTDC time unit to ns
dc31_charge_range = [10,150]
dc31_timing_range = [-60,0]
dc32_charge_range = [10,150]
dc32_timing_range = [-60,0]
TIME_RANGE_NS = (-200, 100) # valid time range for SRPPAC strip hits after anode time subtraction and ns2ns conversion
preamp_type = "asagi" #"rpa"  # "old" or "new"



#spark = SparkSession.builder.getOrCreate()
spark = SparkSession.builder.master("local[10]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
# Register decoder UDFs
spark._jvm.decoders.HRTDCDecoder.registerUDF(spark._jsparkSession)
spark._jvm.decoders.HRTDCUnpairedDecoder.registerUDF(spark._jsparkSession)


runname="1083"
df = spark.read.parquet("/home/h487/notebooks/jan2026/mwdc/parquet/run{}.parquet".format(runname))
df=df.limit(10000)
# Filter SRPPAC anode data and decode
df_sra_w = df.filter("femType==5 and femId==614").select("data").withColumn("decoded",F.expr("decode_hrtdc_segdata(data)"))
df_sra_w = df_sra_w.select("decoded.*").select("hbf.*","data").select("hbfNumber","data").filter("array_size(decoded.data)>0")
df_sra_w = df_sra_w.withColumn("ex",F.explode("data")).select("hbfNumber","ex.*").filter("ch==4") # anode channel is ch=4
df_sra_w = df_sra_w.withColumn("rand",F.rand().cast("float")).withColumn("tcal", (F.col("time").cast("float") + F.col("rand"))*F.lit(CH2NS).cast("float")).drop("rand")
df_sra_w = df_sra_w.withColumn("rand",F.rand().cast("float")).withColumn("sra_charge", (F.col("tot").cast("float") + F.col("rand"))*F.lit(CH2NS).cast("float")).drop("tot").drop("rand")
df_sra_w = df_sra_w.withColumnRenamed("tcal","sra_timing").withColumnRenamed("time","sra_tdc_raw").select("hbfNumber","sra_timing","sra_tdc_raw")
df_sra_w.show(5)
# Select only the fastest sra hit per event
from pyspark.sql import Window
w = Window.partitionBy("hbfNumber").orderBy("sra_timing")
df_sra_w = (
    df_sra_w
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

############ MWDC ######################
df_dc = df.filter("femType==5 and (femId==616 or femId==617 or femId==618)").select("femId","data").withColumn("decoded",F.expr("decode_hrtdc_segdata(data)"))
df_dc = df_dc.select("femId","decoded.*").select("femId","hbf.*","data").select("femId","hbfNumber","data").filter("array_size(decoded.data)>0")
df_dc = df_dc.withColumn("ex",F.explode("data")).select("femId","hbfNumber","ex.*")
df_dc = df_dc.withColumn("rand",F.rand().cast("float")).withColumn("tcal", (F.col("time").cast("float") + F.col("rand"))*F.lit(CH2NS).cast("float")).drop("time").drop("rand")
df_dc = df_dc.withColumn("rand",F.rand().cast("float")).withColumn("charge", (F.col("tot").cast("float") + F.col("rand"))*F.lit(CH2NS).cast("float")).drop("tot").drop("rand")

#Subtract anode timing
df_dc = df_dc.join(df_sra_w, on=["hbfNumber"])
df_dc = df_dc.withColumn("tcal_c", F.expr("tcal-sra_timing")).drop("tcal").drop("sra_timing")

# Map dc31 and dc32
df_map31 = spark.read.csv("../../mwdc/map/dc31_map.csv",inferSchema=True,header=True).withColumn("id",F.col("id").cast("int")).withColumn("femId",F.col("femId").cast("int")).withColumn("ch",F.col("ch").cast("int"))
df_dc31 = df_dc.join(df_map31,on=["ch","femId"]).drop("ch").drop("femId")
df_map32 = spark.read.csv("../../mwdc/map/dc32_map.csv",inferSchema=True,header=True).withColumn("id",F.col("id").cast("int")).withColumn("femId",F.col("femId").cast("int")).withColumn("ch",F.col("ch").cast("int"))
df_dc32 = df_dc.join(df_map32,on=["ch","femId"]).drop("ch").drop("femId")
df_dc31.show(5)
df_dc32.show(5)


# Separate planes and collect hits per event
planes31 = ['x1','x2','y1','y2','x3','x4','y3','y4']
NWIRE = 16
df_planes31 = []
for idx, plane in enumerate(planes31):
    df_plane = df_dc31.filter(f"id>={NWIRE*idx} AND id<{NWIRE*(idx+1)}")
    tname = "dc31_" + plane + "_timing"
    cname = "dc31_" + plane + "_charge"
    iname = "dc31_" + plane + "_id"
    df_plane = df_plane.withColumn(iname, F.col("id")-F.lit(NWIRE*idx))
    df_plane = df_plane.withColumnRenamed("tcal_c",tname)
    df_plane = df_plane.withColumnRenamed("charge",cname)
    df_plane = df_plane.filter(F.expr(f"{cname} > {dc31_charge_range[0]} and {cname} < {dc31_charge_range[1]} and {tname} > {dc31_timing_range[0]} and {tname} < {dc31_timing_range[1]}"))
    df_plane = df_plane.orderBy(F.col(cname).desc()).groupBy("hbfNumber").agg(F.collect_list(iname).alias(iname), F.collect_list(tname).alias(tname), F.collect_list(cname).alias(cname))
    df_planes31.append(df_plane)


planes32 = ['x1','x2','y1','y2']
NWIRE = 16
df_planes32 = []
for idx, plane in enumerate(planes32):
    df_plane = df_dc32.filter(f"id>={NWIRE*idx} AND id<{NWIRE*(idx+1)}")
    tname = "dc32_" + plane + "_timing"
    cname = "dc32_" + plane + "_charge"
    iname = "dc32_" + plane + "_id"
    df_plane = df_plane.withColumn(iname, F.col("id")-F.lit(NWIRE*idx))
    df_plane = df_plane.withColumnRenamed("tcal_c",tname)
    df_plane = df_plane.withColumnRenamed("charge",cname)
    df_plane = df_plane.filter(F.expr(f"{cname} > {dc32_charge_range[0]} and {cname} < {dc32_charge_range[1]} and {tname} > {dc32_timing_range[0]} and {tname} < {dc32_timing_range[1]}"))
    df_plane = df_plane.orderBy(F.col(cname).desc()).groupBy("hbfNumber").agg(F.collect_list(iname).alias(iname), F.collect_list(tname).alias(tname), F.collect_list(cname).alias(cname))
    df_planes32.append(df_plane)    

# Join all planes
df_dc31_planes = df_dc31.select("hbfNumber").dropDuplicates()
for plane in df_planes31:
    df_dc31_planes = df_dc31_planes.join(plane, on=["hbfNumber"])

df_dc32_planes = df_dc32.select("hbfNumber").dropDuplicates()
for plane in df_planes32:
    df_dc32_planes = df_dc32_planes.join(plane, on=["hbfNumber"])

df_mwdc = df_dc31_planes.join(df_dc32_planes, on=["hbfNumber"])
df_mwdc = df_mwdc.join(df_sra_w,on=["hbfNumber"])


rdf = df_mwdc.select("hbfNumber","sra_tdc_raw")
planes = ['dc31_x1','dc31_x2','dc31_y1','dc31_y2','dc31_x3','dc31_x4','dc31_y3','dc31_y4','dc32_x1','dc32_x2','dc32_y1','dc32_y2']
for plane in planes:
    # Create columns for the wire with loargest charge
    dfp = df_mwdc.select("hbfNumber", f"{plane}_id",f"{plane}_charge",f"{plane}_timing")
    dfp = dfp.withColumn("charge0",F.expr(f"element_at({plane}_charge, 1)")) \
             .withColumn("timing0",F.expr(f"element_at({plane}_timing, 1)")) \
             .withColumn("id0",F.expr(f"element_at({plane}_id, 1)"))
    dfp = dfp.select("hbfNumber","id0","charge0","timing0").withColumnRenamed("id0",f"{plane}_id0").withColumnRenamed("charge0",f"{plane}_charge0").withColumnRenamed("timing0",f"{plane}_timing0")
    rdf = rdf.join(dfp, on=["hbfNumber"], how="left")

        
rdf.dropna().show(5)


+---------+----------+-----------+
|hbfNumber|sra_timing|sra_tdc_raw|
+---------+----------+-----------+
|  5859883|  419081.3|  429139276|
|  5859883| 424069.44|  434247118|
|  5859883| 434059.28|  444476710|
|  5859883|  439003.5|  449539592|
|  5859952| 351617.16|  360055976|
+---------+----------+-----------+
only showing top 5 rows
+---------+---------+-----------+---------+---+
|hbfNumber|   charge|sra_tdc_raw|   tcal_c| id|
+---------+---------+-----------+---------+---+
|  5860023| 74.58299|   38480771|252289.34| 73|
|  5860023| 71.21219|   38480771|252293.94|104|
|  5860023|67.712265|   38480771|252296.06|120|
|  5860023| 64.60668|   38480771|252310.88| 90|
|  5860023| 75.03312|   38480771|252289.94|  9|
+---------+---------+-----------+---------+---+
only showing top 5 rows
+---------+---------+-----------+----------+---+
|hbfNumber|   charge|sra_tdc_raw|    tcal_c| id|
+---------+---------+-----------+----------+---+
|  5860023| 80.13981|   38480771| 252297.28| 41|
|  586002

In [ ]:

################## SRPPAC ############################
# Filter SRPPAC anode data and decode
df_sra = df.filter("femType==5 and femId==614").select("data").withColumn("decoded",F.expr("decode_hrtdc_segdata(data)"))
df_sra = df_sra.select("decoded.*").select("hbf.*","data").select("hbfNumber","data").filter("array_size(decoded.data)>0")
df_sra = df_sra.withColumn("ex",F.explode("data")).select("hbfNumber","ex.*").filter("ch==4")
df_sra = df_sra.withColumn("tcal", F.expr(f"(time + rand())*{CH2NS}")).drop("time")
df_sra = df_sra.groupBy("hbfNumber").agg(F.min("tcal").alias("tcal_a")) # select fastest anode hit


df_src = df.filter("femType==7 and femId==615").select("data").withColumn("decoded",F.expr("decode_hrtdc_unpaired_segdata(data)"))
df_src = df_src.select("decoded.*").select("hbf.*","data").select("hbfNumber","data").filter("array_size(decoded.data)>0")
df_src = df_src.withColumn("ex",F.explode("data")).select("hbfNumber","ex.*").withColumnRenamed("tot","edge")
df_src = df_src.withColumn("tcal", F.expr(f"(time + rand())*{CH2NS}")).drop("time")
#df_src.show(5)
# Join strip and anode data
df_sr = df_src.join(df_sra, on=["hbfNumber"])
df_sr = df_sr.withColumn("tcal_c", F.expr("tcal-tcal_a")).drop("tcal").drop("tcal_a")
df_sr = df_sr.filter(f"tcal_c > {TIME_RANGE_NS[0]} and tcal_c < {TIME_RANGE_NS[1]}")


dfL = df_sr.filter("edge=0").select("hbfNumber","tcal_c","ch").withColumnRenamed("tcal_c","timingL")
dfT = df_sr.filter("edge=1").select("hbfNumber","tcal_c","ch").withColumnRenamed("tcal_c","timingT")
df_sr = dfL.join(dfT, on=["hbfNumber","ch"])
df_sr = df_sr.withColumn("charge", F.expr("timingT - timingL"))


df_xmap = spark.read.csv(f'/home/h487/notebooks/jan2026/srppac/map/srx_{preamp_type}_map.csv', inferSchema = True, header = True).withColumn("id", F.col("id").cast("int"))
df_ymap = spark.read.csv(f'/home/h487/notebooks/jan2026/srppac/map/sry_{preamp_type}_map.csv', inferSchema = True, header = True).withColumn("id", F.col("id").cast("int"))
df_prm_x = spark.read.csv(f'/home/h487/notebooks/jan2026/srppac/prm/srx_charge_calib_{preamp_type}.csv', inferSchema = True, header = True)
df_prm_y = spark.read.csv(f'/home/h487/notebooks/jan2026/srppac/prm/sry_charge_calib_{preamp_type}.csv', inferSchema = True, header = True)


dfs = [(df_xmap, df_prm_x), (df_ymap, df_prm_y)]
df_xy_list = []
for df_map, df_prm in dfs:
    df_xy = df_sr.join(df_map, on = ["ch"]).drop("ch")
    #df_xy = df_xy.join(df_prm, on=["id"], how="left")
    #df_xy = df_xy.withColumn("charge", F.expr("charge * p1 + p0")).drop("p1").drop("p0")

    df_xy = df_xy.orderBy(F.col("charge").desc()).groupBy("hbfNumber").agg(F.collect_list("id").alias("id"), F.collect_list("timingL").alias("timing"), F.collect_list("charge").alias("charge"))
    df_xy = df_xy.withColumn("size", F.expr("size(id)"))
    df_xy = df_xy.withColumn("timing0",F.expr(f"try_element_at(timing, 1)")) \
                 .withColumn("charge0",F.expr(f"try_element_at(charge, 1)")) \
                 .withColumn("charge1",F.expr(f"try_element_at(charge, 2)")) \
                 .withColumn("charge2",F.expr(f"try_element_at(charge, 3)")) \
                 .withColumn("id0",F.expr(f"try_element_at(id, 1)")) \
                 .withColumn("id1",F.expr(f"try_element_at(id, 2)")) \
                 .withColumn("id2",F.expr(f"try_element_at(id, 3)")) \
                 .withColumn("q0q1", F.expr("(charge0-charge1)/(charge0+charge1)"))
        
    df_xy_list.append(df_xy)

# Add suffix _y to all columns except hbfNumber
df_xy_list[0] = df_xy_list[0].select([F.col(col).alias(f"{col}_x") if col != "hbfNumber" else F.col(col) for col in df_xy_list[0].columns])
df_xy_list[1] = df_xy_list[1].select([F.col(col).alias(f"{col}_y") if col != "hbfNumber" else F.col(col) for col in df_xy_list[1].columns])
srppacdf = df_xy_list[0].join(df_xy_list[1], on=["hbfNumber"], how="outer")
srppacdf=srppacdf.filter("size_x>0 AND size_x<6").filter("size_y>0 AND size_y<6")
srppacdf.show(5,truncate=False)


+---------+--------------------+---------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+------+------------------+------------------+------------------+-------------------+-----+-----+-----+-------------------+--------------------+----------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+------+-------------------+------------------+------------------+-------------------+-----+-----+-----+-------------------+
|hbfNumber|id_x                |timing_x                                                                                           |charge_x                                                                                            |size_x|timing0_x         |charge0_x         |charge1_x         |ch

In [8]:
HIT_SIZE_LIMIT = 6
planes = ["x","y"]
xyrdf = srppacdf.filter(f"size_x<{HIT_SIZE_LIMIT} and size_y<{HIT_SIZE_LIMIT}")
xyrdf = xyrdf.filter("size_x>0 AND size_y>0")
# Charge0 and Charge1 strips should be adjacent
xyrdf = xyrdf.filter("ABS(id0_x - id1_x) = 1 AND ABS(id0_y - id1_y) = 1")

xydfs = [
    xyrdf.select("hbfNumber","size_x","id0_x","id1_x","charge0_x","charge1_x").withColumnRenamed("id0_x","id0").withColumnRenamed("id1_x","id1").withColumnRenamed("charge0_x","charge0").withColumnRenamed("charge1_x","charge1"),
    xyrdf.select("hbfNumber","size_y","id0_y","id1_y","charge0_y","charge1_y").withColumnRenamed("id0_y","id0").withColumnRenamed("id1_y","id1").withColumnRenamed("charge0_y","charge0").withColumnRenamed("charge1_y","charge1")
    ]

xydfs[0].show(5)
xydfs[1].show(5)

df_join = xydfs[0].join(rdf, on=["hbfNumber"]).dropDuplicates().dropna().orderBy("hbfNumber")


+---------+------+---+---+-------+-------+
|hbfNumber|size_x|id0|id1|charge0|charge1|
+---------+------+---+---+-------+-------+
+---------+------+---+---+-------+-------+

+---------+------+---+---+-------+-------+
|hbfNumber|size_y|id0|id1|charge0|charge1|
+---------+------+---+---+-------+-------+
+---------+------+---+---+-------+-------+



In [9]:

df_join.filter("id0==20").show(20)
#df_join.show(20)

+---------+------+---+---+-------+-------+-----------+-----------+---------------+---------------+-----------+---------------+---------------+-----------+---------------+---------------+-----------+---------------+---------------+-----------+---------------+---------------+-----------+---------------+---------------+-----------+---------------+---------------+-----------+---------------+---------------+-----------+---------------+---------------+-----------+---------------+---------------+-----------+---------------+---------------+-----------+---------------+---------------+
|hbfNumber|size_x|id0|id1|charge0|charge1|sra_tdc_raw|dc31_x1_id0|dc31_x1_charge0|dc31_x1_timing0|dc31_x2_id0|dc31_x2_charge0|dc31_x2_timing0|dc31_y1_id0|dc31_y1_charge0|dc31_y1_timing0|dc31_y2_id0|dc31_y2_charge0|dc31_y2_timing0|dc31_x3_id0|dc31_x3_charge0|dc31_x3_timing0|dc31_x4_id0|dc31_x4_charge0|dc31_x4_timing0|dc31_y3_id0|dc31_y3_charge0|dc31_y3_timing0|dc31_y4_id0|dc31_y4_charge0|dc31_y4_timing0|dc32_x1_id0

In [ ]:
df_join.filter("id0==20").show(20)

In [ ]:
from hist.sparkHist2d import Hist2D
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm

plt.figure(0, figsize=(12,8))
plt.rcParams["font.size"] = 8
plt.subplot2grid((2,1),(0,0))
h = Hist2D(df_dc31, ["id","charge"], [125,200], [[-0.5, 123.5], [0, 200]], norm=LogNorm(), interpolation='none')
plt.show()

## Plot charge vs id histogram for dc32

In [ ]:
from hist.sparkHist2d import Hist2D
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm

plt.figure(0, figsize=(12,8))
plt.rcParams["font.size"] = 8
#plt.subplot2grid((2,1),(0,0))
#h = Hist2D(df_dc31, ["id","charge"], [125,200], [[-0.5, 123.5], [0, 200]], norm=LogNorm(), interpolation='none')
plt.subplot2grid((2,1),(1,0))
h = Hist2D(df_dc32, ["id","charge"], [65,200], [[-0.5, 63.5], [0, 200]], norm=LogNorm(), interpolation='none')
plt.show()